# 06 — Robustesse de l'évaluation

Nos chiffres viennent d'une validation croisée en 5 plis **par station**. Ce notebook pose trois questions sur cette
évaluation elle-même :

1. **Quelle est la marge d'incertitude ?** 419 paires, ce n'est pas énorme : 75 % de paires EPA, c'est 75 % à combien
   près ? Et l'écart entre le modèle combiné et le réglage 1 est-il réel ?
2. **Les plis par station flattent-ils les résultats ?** Deux stations d'une même ville voient le même air. Si l'une
   est en apprentissage et l'autre en test, le modèle a déjà vu les épisodes qu'il doit prédire.
3. **Le modèle tient-il dans le temps ?** Jusqu'ici, il apprend sur 2024 et 2025 à la fois. Que se passe-t-il s'il
   apprend sur une année et doit corriger l'année suivante ?

**Pré-requis.** `make data` puis `make train` (environ une minute d'entraînement dans ce notebook). LightGBM n'est pas
parfaitement reproductible quand il calcule sur plusieurs cœurs : d'une exécution à l'autre, les pourcentages
peuvent bouger de quelques dixièmes.

In [1]:
import polars as pl

from aircalib import modeles as M
from aircalib import parametres as P
from aircalib import variables as V
from aircalib.metriques import resume, tirages_par_station

pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_width_chars(150)
DONNEES = P.dossier_donnees()
SANS_VOISINS = [v for v in V.VARIABLES if v not in V.FAMILLES["voisins"]]
COLONNES = ["rmse", "r2_median_par_paire", "pente_mediane_par_paire", "pct_epa_horaire", "pct_epa_journalier"]

table = pl.read_parquet(DONNEES / "variables" / "table.parquet")
reference = {
    nom: pl.read_parquet(DONNEES / "predictions" / f"{nom}.parquet").filter(pl.col("paire_saine"))
    for nom in ["5_lightgbm_combine", "3_lightgbm_reglage_1"]
}


def modele_combine(table: pl.DataFrame, apprendre_sur=None, tester_sur=None) -> pl.DataFrame:
    """Le modèle retenu, réentraîné avec d'autres plis ou d'autres années (paires saines seulement)."""
    avec = M.validation_croisee(table, M.lightgbm(V.VARIABLES), apprendre_sur, tester_sur)
    sans = M.validation_croisee(table, M.lightgbm(SANS_VOISINS), apprendre_sur, tester_sur)
    return M.combiner(avec, sans).filter(pl.col("paire_saine"))


def ligne(nom: str, predictions: pl.DataFrame) -> dict:
    chiffres = resume(predictions)
    return {"essai": nom, "paires": chiffres["paires_notees"]} | {c: round(chiffres[c], 3) for c in COLONNES}

## 1. Intervalles de confiance

On fait un **bootstrap par station** : on retire 179 stations au hasard parmi les 179, avec remise (certaines
reviennent deux fois, d'autres pas du tout), et on recalcule les chiffres. 1 000 fois. L'intervalle à 95 % va du
2,5ᵉ au 97,5ᵉ centile des tirages.

On tire des stations et pas des heures : les heures d'une même paire se ressemblent énormément, et les traiter
comme indépendantes donnerait une précision illusoire. Pour comparer deux modèles, on garde les mêmes tirages
pour les deux et on regarde la différence tirage par tirage.

In [2]:
tirages = {nom: tirages_par_station(pred) for nom, pred in reference.items()}
tirages["écart combiné − réglage 1"] = tirages["5_lightgbm_combine"] - tirages["3_lightgbm_reglage_1"]
pl.DataFrame(
    [
        {
            "": nom,
            **{
                f"{c} [2,5 % ; 97,5 %]": f"{t[c].quantile(0.025):.2f} ; {t[c].quantile(0.975):.2f}"
                for c in ["rmse", "pct_epa_horaire"]
            },
        }
        for nom, t in tirages.items()
    ]
)

,"rmse [2,5 % ; 97,5 %]","pct_epa_horaire [2,5 % ; 97,5 %]"
str,str,str
"""5_lightgbm_combine""","""3.50 ; 4.02""","""68.70 ; 81.89"""
"""3_lightgbm_reglage_1""","""3.48 ; 4.01""","""68.20 ; 82.14"""
"""écart combiné − réglage 1""","""-0.01 ; 0.04""","""-1.45 ; 1.70"""


- Le modèle combiné passe les cibles EPA pour **75,7 % des paires, avec un intervalle d'environ 69 à 82 %**. Le
  RMSE est plus stable : de 3,5 à 4,0 µg/m³.
- L'écart entre le combiné et le réglage 1 va d'environ −1,5 à +1,7 point : **il contient zéro**. Les deux
  modèles ne sont pas distinguables avec nos données. C'est ce qu'on disait dans le README (« les deux se
  valent »), mais maintenant c'est mesuré.
- Autrement dit, nos pourcentages sont précis à environ ± 6 à 7 points : les écarts de un ou deux points entre modèles,
  dans les notebooks 03 et 04, ne sont pas significatifs.

## 2. Des plis par région

On regroupe les stations à 10 km ou moins les unes des autres (de proche en proche) en **régions**, et on met
chaque région entière dans un seul pli. La position d'une station est approchée par la moyenne de ses capteurs
(tous à 1,5 km ou moins d'elle).

In [3]:
positions = (
    pl.read_parquet(DONNEES / "propre" / "paires.parquet")
    .filter(pl.col("capteur").is_in(table["capteur"].unique().implode()))
    .group_by("station")
    .agg(pl.col("lat").mean(), pl.col("lon").mean())
)
zones = V.regions(positions)
taille = zones.group_by("region").len()
print(
    f"{zones.height} stations en {taille.height} régions ; "
    f"{(taille['len'] > 1).sum()} régions ont plusieurs stations (jusqu'à {taille['len'].max()})"
)
par_region = modele_combine(V.plis_par_region(table, zones))
pl.DataFrame(
    [ligne("plis par station", reference["5_lightgbm_combine"]), ligne("plis par région", par_region)]
)

179 stations en 107 régions ; 41 régions ont plusieurs stations (jusqu'à 8)


essai,paires,rmse,r2_median_par_paire,pente_mediane_par_paire,pct_epa_horaire,pct_epa_journalier
str,i64,f64,f64,f64,f64,f64
"""plis par station""",419,3.744,0.851,0.803,75.656,86.52
"""plis par région""",419,3.819,0.851,0.801,72.076,85.049


Avec des plis par région, le modèle perd **environ 3,5 points de paires EPA en horaire** (de 75,7 à 72,1 %), 1,5 point
en journalier, et moins de 0,1 µg/m³ de RMSE. (LightGBM n'étant pas parfaitement reproductible, ce chiffre bouge
entre 3 et 4 points d'une exécution à l'autre.) Les plis par station flattaient donc un peu les résultats, mais l'écart
reste dans l'intervalle de confiance de la partie 1. La conclusion du README tient : la station voisine aide un peu,
elle n'explique pas le résultat.

## 3. Dans le temps

On garde les plis par station (le test est toujours sur des stations jamais vues), mais on apprend sur une année
et on teste sur l'autre. Pour séparer l'effet du temps de celui d'une base d'apprentissage deux fois plus petite,
on compare à un essai **témoin** qui apprend et teste sur la même année, avec la même quantité de données.

In [4]:
annee = pl.col("heure").dt.year()
essais = {
    "2024 → 2025": (annee == 2024, annee == 2025),
    "témoin 2025 → 2025": (annee == 2025, annee == 2025),
    "2025 → 2024": (annee == 2025, annee == 2024),
    "témoin 2024 → 2024": (annee == 2024, annee == 2024),
}
pl.DataFrame([ligne(nom, modele_combine(table, *filtres)) for nom, filtres in essais.items()])

essai,paires,rmse,r2_median_par_paire,pente_mediane_par_paire,pct_epa_horaire,pct_epa_journalier
str,i64,f64,f64,f64,f64,f64
"""2024 → 2025""",332,4.357,0.831,0.699,59.639,69.018
"""témoin 2025 → 2025""",332,3.77,0.876,0.82,82.831,90.491
"""2025 → 2024""",375,4.058,0.78,0.761,64.533,75.137
"""témoin 2024 → 2024""",375,3.543,0.844,0.811,74.933,86.066


C'est le résultat le plus important de ce notebook. **Appris sur une année, le modèle passe nettement moins bien
les cibles l'année suivante** : environ 60 % de paires EPA au lieu de 83 % (2024 → 2025), et 65 % au lieu de 75 %
(2025 → 2024). La pente médiane baisse aussi (0,70 et 0,76 au lieu de 0,82 et 0,81) : les pics sont encore
plus sous-estimés.

Deux explications se mélangent, et on ne peut pas les séparer ici :
- **Le temps change vraiment les choses** : les capteurs vieillissent, la météo et la composition des particules
  changent d'une année à l'autre.
- **Dans la validation par station, le modèle a vu les mêmes jours ailleurs.** Un épisode de pollution touche
  toute une région : pendant l'apprentissage, le modèle a vu ce jour-là sur d'autres stations. Ça ne triche pas
  sur la station testée, mais ça l'aide.

**Ce que ça change pour l'usage.** Nos 75 % décrivent un modèle **réappris régulièrement** sur les stations de la
même période, pas un modèle appris une fois pour toutes. Si on le fige, il faut s'attendre à quelque chose comme
60 à 65 % de paires EPA un an plus tard.

## Conclusion

| question | réponse |
|---|---|
| Précision de nos chiffres | 75,7 % de paires EPA, intervalle à 95 % d'environ 69–82 % |
| Combiné ou réglage 1 ? | indistinguables (l'écart contient zéro) |
| Plis par station trop optimistes ? | un peu : environ −3 à −4 points avec des plis par région |
| Le modèle tient-il un an ? | pas tel quel : de −10 à −23 points, il faut le réapprendre |